# Week 6: Single-Cell RNA-seq Analysis Pipeline

This notebook implements a complete single-cell RNA-seq analysis pipeline:
1. **Alignment & Quantification**: Using Alevin-fry (via simpleaf) to process FASTQ files
2. **Clustering**: Using Leiden algorithm with UMAP visualization
3. **Cell Type Annotation**: Using CellTypist for automatic annotation

## Data
- Sample data: Human glioblastoma single-cell data (subset mapped to chr5)
- Reference: Human genome chromosome 5 with GTF annotations
- Chemistry: 10x Chromium v3

---
## Part 0: Environment Setup
Install all required dependencies using conda/mamba.

In [ ]:
%%bash
# Install mamba for faster conda operations (if not already installed)
if ! command -v mamba &> /dev/null; then
    conda install -y -c conda-forge mamba
fi

In [ ]:
%%bash
# Install simpleaf and alevin-fry for alignment/quantification
mamba install -y -c bioconda -c conda-forge simpleaf

In [ ]:
%%bash
# Install Python packages for analysis
pip install scanpy pyroe celltypist leidenalg

---
## Part 1: Data Download and Preparation
Download the FASTQ files, reference genome, and whitelist barcodes.

In [ ]:
%%bash
# Create working directory
mkdir -p sc_analysis
cd sc_analysis

# Download and extract the sample data (FASTQ + reference)
echo "Downloading sample data..."
wget -q -O toy_read_ref_set.tar.gz "https://app.box.com/shared/static/lx2xownlrhz3us8496tyu9c4dgade814.gz"
tar -xzf toy_read_ref_set.tar.gz

# Download the 10x v3 whitelist barcode file
echo "Downloading whitelist barcodes..."
wget -q -O 3M-february-2018.txt.gz "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
gunzip -f 3M-february-2018.txt.gz

echo "Download complete!"
ls -la
ls -la toy_ref_read/

In [ ]:
%%bash
cd sc_analysis

# Verify the data structure
echo "=== Reference files ==="
ls -la toy_ref_read/toy_human_ref/fasta/
ls -la toy_ref_read/toy_human_ref/genes/

echo ""
echo "=== FASTQ files ==="
ls -la toy_ref_read/toy_read_fastq/

echo ""
echo "=== Whitelist ==="
wc -l 3M-february-2018.txt

---
## Part 2: Alignment and Quantification with Alevin-fry

We use `simpleaf` which wraps the alevin-fry pipeline:
1. Build a splici index (spliced transcripts + introns)
2. Map reads and quantify gene expression

In [ ]:
%%bash
cd sc_analysis

# Set up alevin-fry home directory
mkdir -p alevin_fry_home
export ALEVIN_FRY_HOME="$(pwd)/alevin_fry_home"

# Configure simpleaf
simpleaf set-paths

echo "Simpleaf configured successfully!"

In [ ]:
%%bash
cd sc_analysis
export ALEVIN_FRY_HOME="$(pwd)/alevin_fry_home"

# Step 1: Build the splici index
# -f: genome FASTA file
# -g: gene annotation GTF file  
# -r: read length (90bp based on R2 reads)
# -t: number of threads

echo "Building splici index..."
simpleaf index \
    -o simpleaf_index \
    -f toy_ref_read/toy_human_ref/fasta/genome.fa \
    -g toy_ref_read/toy_human_ref/genes/genes.gtf \
    -r 90 \
    -t 4

echo "Index built successfully!"
ls -la simpleaf_index/

In [ ]:
%%bash
cd sc_analysis
export ALEVIN_FRY_HOME="$(pwd)/alevin_fry_home"

# Step 2: Quantification
# -c: chemistry (10xv3)
# -1: R1 reads (barcode + UMI)
# -2: R2 reads (biological sequence)
# -i: index directory
# -u: use unfiltered permit list (knee finding)
# -r: resolution method
# -m: transcript-to-gene mapping

echo "Running quantification..."
simpleaf quant \
    -c 10xv3 \
    -t 4 \
    -1 toy_ref_read/toy_read_fastq/selected_R1_reads.fastq \
    -2 toy_ref_read/toy_read_fastq/selected_R2_reads.fastq \
    -i simpleaf_index/index \
    -u 3M-february-2018.txt \
    -r cr-like \
    -m simpleaf_index/index/t2g_3col.tsv \
    -o simpleaf_quant

echo "Quantification complete!"
ls -la simpleaf_quant/

In [ ]:
%%bash
cd sc_analysis

# Examine the output
echo "=== Quantification output ==="
ls -la simpleaf_quant/af_quant/alevin/

echo ""
echo "=== Sample of count matrix ==="
head -20 simpleaf_quant/af_quant/alevin/quants_mat.mtx

echo ""
echo "=== Number of cells ==="
wc -l simpleaf_quant/af_quant/alevin/quants_mat_rows.txt

echo ""
echo "=== Number of genes ==="
wc -l simpleaf_quant/af_quant/alevin/quants_mat_cols.txt

---
## Part 3: Load Data and Create AnnData Object

Load the quantified data into Python using pyroe and create an AnnData object for downstream analysis with scanpy.

In [ ]:
import scanpy as sc
import pyroe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')

print(f"Scanpy version: {sc.__version__}")

In [ ]:
# Load the alevin-fry quantification results
# Using USA mode: sum of Unspliced, Spliced, and Ambiguous counts
quant_dir = 'sc_analysis/simpleaf_quant/af_quant'

adata = pyroe.load_fry(quant_dir, output_format={'X': ['U', 'S', 'A']})

print(f"Loaded AnnData object:")
print(f"  Number of cells: {adata.n_obs}")
print(f"  Number of genes: {adata.n_vars}")
print(adata)

In [ ]:
# Basic QC metrics
print("=== Basic QC ===")
print(f"Total counts: {adata.X.sum():.0f}")
print(f"Mean counts per cell: {adata.X.sum(axis=1).mean():.2f}")
print(f"Mean genes per cell: {(adata.X > 0).sum(axis=1).mean():.2f}")

---
## Part 4: Preprocessing

Standard single-cell preprocessing steps:
1. Filter cells and genes
2. Normalize and log-transform
3. Identify highly variable genes
4. Scale data
5. PCA for dimensionality reduction

In [ ]:
# Make a copy of raw counts
adata.layers['counts'] = adata.X.copy()

# Basic filtering
print("Before filtering:")
print(f"  Cells: {adata.n_obs}, Genes: {adata.n_vars}")

# Filter cells with minimum counts and genes
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_cells(adata, min_genes=5)

# Filter genes expressed in minimum number of cells
sc.pp.filter_genes(adata, min_cells=1)

print("After filtering:")
print(f"  Cells: {adata.n_obs}, Genes: {adata.n_vars}")

In [ ]:
# Normalize to median total counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

# Store normalized data
adata.raw = adata.copy()

print("Normalization complete!")

In [ ]:
# Identify highly variable genes
# Using a lower threshold due to small dataset
n_top_genes = min(500, adata.n_vars - 1)

if adata.n_vars > 10:
    sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, flavor='seurat')
    print(f"Identified {adata.var['highly_variable'].sum()} highly variable genes")
else:
    # If very few genes, mark all as highly variable
    adata.var['highly_variable'] = True
    print(f"Small gene set - using all {adata.n_vars} genes")

In [ ]:
# Scale data
sc.pp.scale(adata, max_value=10)

# PCA
n_comps = min(50, adata.n_obs - 1, adata.n_vars - 1)
sc.tl.pca(adata, n_comps=n_comps)

print(f"PCA computed with {n_comps} components")

In [ ]:
# Plot PCA variance ratio
if adata.n_obs > 2:
    sc.pl.pca_variance_ratio(adata, n_pcs=min(30, n_comps), show=True)

---
## Part 5: Clustering with Leiden Algorithm

1. Compute neighborhood graph
2. Run Leiden clustering
3. Compute UMAP embedding
4. Visualize clusters

In [ ]:
# Compute neighborhood graph
n_neighbors = min(15, adata.n_obs - 1)
n_pcs = min(20, n_comps)

sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)

print(f"Computed neighbors graph (n_neighbors={n_neighbors}, n_pcs={n_pcs})")

In [ ]:
# Run Leiden clustering
sc.tl.leiden(adata, resolution=0.5)

print(f"Leiden clustering complete!")
print(f"Number of clusters: {adata.obs['leiden'].nunique()}")
print(f"\nCluster sizes:")
print(adata.obs['leiden'].value_counts().sort_index())

In [ ]:
# Compute UMAP embedding
sc.tl.umap(adata)

print("UMAP embedding computed!")

In [ ]:
# Plot UMAP with Leiden clusters
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(adata, color='leiden', ax=ax, show=False, 
           title='Leiden Clustering', legend_loc='on data')
plt.tight_layout()
plt.savefig('sc_analysis/clustering_umap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Clustering plot saved to: sc_analysis/clustering_umap.png")

---
## Part 6: Cell Type Annotation with CellTypist

Use CellTypist for automatic cell type annotation based on reference models.

In [ ]:
import celltypist
from celltypist import models

# Download available models
models.download_models(force_update=False)

# List available models
print("Available CellTypist models:")
print(models.models_description())

In [ ]:
# Load a general human immune model
# Using Immune_All_Low for broader cell type categories
model = models.Model.load(model='Immune_All_Low.pkl')

print(f"Model loaded: {model.description}")
print(f"Number of cell types: {len(model.cell_types)}")

In [ ]:
# Prepare data for CellTypist (needs raw normalized counts, not scaled)
# CellTypist expects log1p normalized data
adata_for_celltypist = adata.raw.to_adata().copy()

print(f"Data prepared for CellTypist:")
print(f"  Cells: {adata_for_celltypist.n_obs}")
print(f"  Genes: {adata_for_celltypist.n_vars}")

In [ ]:
# Run CellTypist prediction
predictions = celltypist.annotate(adata_for_celltypist, model=model, majority_voting=True)

print("CellTypist annotation complete!")

In [ ]:
# Get the annotated AnnData
adata_annotated = predictions.to_adata()

# Transfer annotations to our main adata object
adata.obs['cell_type_predicted'] = adata_annotated.obs['predicted_labels']
adata.obs['cell_type_majority'] = adata_annotated.obs['majority_voting']
adata.obs['cell_type_conf'] = adata_annotated.obs['conf_score']

print("\n=== Predicted Cell Types (per cell) ===")
print(adata.obs['cell_type_predicted'].value_counts())

print("\n=== Cell Types (majority voting) ===")
print(adata.obs['cell_type_majority'].value_counts())

In [ ]:
# Plot UMAP with cell type annotations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Leiden clusters
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False,
           title='Leiden Clusters', legend_loc='on data')

# Plot 2: Cell type annotations
sc.pl.umap(adata, color='cell_type_majority', ax=axes[1], show=False,
           title='CellTypist Annotation (Majority Voting)', 
           legend_loc='right margin')

plt.tight_layout()
plt.savefig('sc_analysis/annotation_umap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Annotation plot saved to: sc_analysis/annotation_umap.png")

In [ ]:
# Summary table: cluster vs cell type
print("=== Cluster vs Cell Type Summary ===")
ct_summary = pd.crosstab(adata.obs['leiden'], adata.obs['cell_type_majority'])
print(ct_summary)

In [ ]:
# Additional visualization: dotplot of cell types
fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.umap(adata, color=['leiden', 'cell_type_majority', 'cell_type_conf'],
           ncols=3, show=True)
plt.savefig('sc_analysis/full_annotation_panel.png', dpi=150, bbox_inches='tight')
print("Full annotation panel saved!")

---
## Summary

This notebook completed the following single-cell RNA-seq analysis pipeline:

### 1. Alignment & Quantification (1 point)
- Downloaded sample FASTQ files and reference genome (human chr5)
- Built splici index using simpleaf/alevin-fry
- Quantified gene expression using 10x Chromium v3 chemistry

### 2. Clustering (1 point)
- Loaded count matrix into AnnData object
- Performed standard preprocessing (filtering, normalization, HVG selection, PCA)
- Computed neighborhood graph and UMAP embedding
- Performed Leiden clustering

### 3. Cell Type Annotation (2 points)
- Used CellTypist with Immune_All_Low model
- Annotated cells with predicted cell types
- Visualized annotations on UMAP plot

In [ ]:
# Final summary
print("="*50)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("="*50)
print(f"\nFinal dataset:")
print(f"  Cells: {adata.n_obs}")
print(f"  Genes: {adata.n_vars}")
print(f"  Clusters: {adata.obs['leiden'].nunique()}")
print(f"  Cell types identified: {adata.obs['cell_type_majority'].nunique()}")
print(f"\nOutput files:")
print(f"  - sc_analysis/clustering_umap.png")
print(f"  - sc_analysis/annotation_umap.png")
print(f"  - sc_analysis/full_annotation_panel.png")